# Shot-model regularisation selection and test evaluation

The experiment selects the L2 coefficient using **2023–2024 training and 2025
validation**, refits the whole pipeline on **2023–2025**, then evaluates the
frozen model on **2026**. Weighted validation binary cross-entropy selects lambda.
The classification threshold stays at 0.5.

This notebook reads saved experiment artifacts. It does not tune, refit or
rescore the test set when rerun. Run `python -m afl_opposition.tuning` once after
preparing all four seasons to generate those artifacts. An explicit
`--overwrite` is required to repeat an existing experiment.

Run cells in order using the project kernel. Saved outputs document the completed run.
The [experiment specification](../docs/tuned-logistic.md) gives the complete
protocol, results and limitations. Earlier notebooks retain their development
results; their statements that 2026 was unscored describe those earlier stages.

## 1. Experiment records

Experiment paths are resolved relative to the repository root.
The artifact folder contains separate selection and test reports, the lambda
search, and a refit record written before loading the test examples.

In [1]:
import hashlib
import json
from pathlib import Path

import pandas as pd
from IPython.display import display

project_root = next(
    (p for p in [Path.cwd(), *Path.cwd().parents]
     if (p / "data/sources.json").is_file() and (p / "pyproject.toml").is_file()), None,
)
if project_root is None:
    raise RuntimeError("Launch from the repository root or notebooks folder.")
experiment = project_root / "artifacts/logistic-tuned-2026"
if not (experiment / "test/summary.json").is_file():
    raise RuntimeError("Run python -m afl_opposition.tuning after preparing the four seasons.")
trials = pd.read_parquet(experiment / "lambda_search.parquet")
selection = json.loads((experiment / "selection/summary.json").read_text())
refit = json.loads((experiment / "refit.json").read_text())
test = json.loads((experiment / "test/summary.json").read_text())
print("Experiment:", experiment.relative_to(project_root).as_posix())

Experiment: artifacts/logistic-tuned-2026


## 2. Regularisation coefficient and selection objective

Lambda is the coefficient of the squared coefficient norm:

$$J=\frac{1}{S}\sum_i s_i\operatorname{BCE}(y_i,p_i)+\lambda\|\beta\|_2^2.$$

The intercept is unpenalised. Each possession receives total weight one across
its eligible prefixes. Scikit-learn's equivalent setting is
$C=1/(2\lambda S)$, where $S=\sum_i s_i$. For $\lambda=0$, `C` is infinite and
the model is unregularised. The zero candidate checks that boundary explicitly.

Every candidate fits on 2023–2024 only, using an encoder and scaler fitted on
that same partition. Selection minimises **weighted 2025 BCE without the L2
penalty**. Validation scores measure predictive error; the penalty enters fitting.
AP and Brier score are recorded as diagnostics, not selection criteria.

The coarse search tests zero and powers of ten from $10^{-8}$ to $10^1$.
A predefined boundary check and quarter-decade refinement use validation only.
Exact validation-loss ties favour larger lambda. The result is the best tested
value, not proof of a global or statistically unique optimum.

In [2]:
display(pd.DataFrame({
    "setting": ["Training seasons", "Validation season", "Selection criterion", "Threshold"],
    "value": [str(selection["training_seasons"]), str(selection["validation_season"]),
              selection["selection_metric"], str(selection["decision_threshold"])],
}))
print("Coarse lambdas:", selection["coarse_lambdas"])

,setting,value
0,Training seasons,"[2023, 2024]"
1,Validation season,2025
2,Selection criterion,minimum possession-weighted 2025 binary cross-...
3,Threshold,0.5


Coarse lambdas: [0.0, 1e-08, 1e-07, 1e-06, 1e-05, 0.0001, 0.001, 0.01, 0.1, 1.0, 10.0]


## 3. Regularisation search results

The full search table reports training and validation BCE across penalty
strengths. The `selected` column identifies the chosen configuration. Lambda zero is displayed explicitly. The tolerance
is tighter than in the initial demonstration baseline to resolve small changes
in the validation loss.

Differences in the final decimal places do not establish a meaningful
performance advantage. The validation set has been used for model selection;
its minimum loss is not the final generalisation estimate.

In [3]:
display(trials.sort_values("lambda_l2")[[
    "lambda_l2", "C", "stage", "training_log_loss", "validation_log_loss",
    "validation_average_precision", "validation_brier_score", "selected",
]].style.format({"lambda_l2": "{:.8g}", "C": "{:.8g}", "training_log_loss": "{:.9f}",
                 "validation_log_loss": "{:.9f}", "validation_average_precision": "{:.6f}",
                 "validation_brier_score": "{:.6f}"}))
best = trials.loc[trials.selected].iloc[0]
assert int(trials.selected.sum()) == 1
assert best.validation_log_loss == trials.validation_log_loss.min()
print("Selected lambda:", best.lambda_l2)
print("Best validation BCE:", best.validation_log_loss)

,lambda_l2,C,stage,training_log_loss,validation_log_loss,validation_average_precision,validation_brier_score,selected
0,0,inf,coarse,0.399827623,0.411178697,0.538694,0.129450,False
1,1e-08,485.06956,coarse,0.399827623,0.411178693,0.538694,0.129450,False
2,1e-07,48.506956,coarse,0.399827623,0.411178690,0.538694,0.129451,False
11,1.7782794e-07,27.277466,refined,0.399827623,0.411178677,0.538694,0.129451,False
12,3.1622777e-07,15.339246,refined,0.399827624,0.411178659,0.538693,0.129451,False
13,5.6234133e-07,8.6258921,refined,0.399827627,0.411178634,0.538693,0.129451,False
3,1e-06,4.8506956,coarse,0.399827636,0.411178592,0.538692,0.129451,False
14,1.7782794e-06,2.7277466,refined,0.399827666,0.411178533,0.538690,0.129451,False
15,3.1622777e-06,1.5339246,refined,0.399827755,0.411178460,0.538687,0.129451,False
16,5.6234133e-06,0.86258921,refined,0.399828027,0.411178436,0.538682,0.129451,True


Selected lambda: 5.623413251903491e-06
Best validation BCE: 0.4111784362719436


## 4. Refitting with the selected regularisation

The final model starts from a fresh encoder, scaler and classifier and fits all
eligible 2023–2025 examples. Validation is now part of the fitting data, so it
no longer supplies an independent evaluation of this refitted model.

The selected lambda stays fixed. For positive lambda, `C` decreases when the
total sample weight increases, preserving the coefficient of the L2 term in
the mean-loss objective. Reusing the old `C` would weaken that coefficient.
An unregularised selection keeps `C` infinite.

In [4]:
display(pd.DataFrame([
    {"stage": "Lambda selection fit", "seasons": "2023–2024", "lambda_l2": selection["lambda_l2"],
     "weight_sum": selection["training_weight_sum"], "C": selection["C"]},
    {"stage": "Final refit", "seasons": "2023–2025", "lambda_l2": refit["lambda_l2"],
     "weight_sum": refit["training_weight_sum"], "C": refit["C"]},
]))
assert refit["lambda_l2"] == selection["lambda_l2"]
if refit["lambda_l2"] > 0:
    assert abs(1 / (2 * refit["C"] * refit["training_weight_sum"]) - refit["lambda_l2"]) < 1e-12
print("Refit matches:", refit["refit_matches"])
print("Refit possessions:", refit["refit_possessions"])
print("Refit action states:", refit["refit_examples"])

,stage,seasons,lambda_l2,weight_sum,C
0,Lambda selection fit,2023–2024,0.000006,103078.0,0.862589
1,Final refit,2023–2025,0.000006,153822.0,0.578032


Refit matches: 647
Refit possessions: 153822
Refit action states: 844775


## 5. Frozen-model verification and test coverage

The refit file and model were saved before the test loader was called. Their
hashes connect the test report to that frozen model. Test scoring applies the
existing timestamp, action and label-quality policies without fitting any
component or changing lambda. The same 0.5 threshold is retained.

The source audit exposes exclusions. These results concern eligible archived
possessions, not every action or possession in the season. The test is now
observed; subsequent choices based on its results would make it development
evidence rather than a fresh holdout.

In [5]:
assert hashlib.sha256((experiment / "refit_model.pkl").read_bytes()).hexdigest() == refit["model_sha256"]
assert hashlib.sha256((experiment / "refit.json").read_bytes()).hexdigest() == test["refit_sha256"]
assert hashlib.sha256((experiment / "lambda_search.parquet").read_bytes()).hexdigest() == refit["lambda_search_sha256"]
print("Frozen before test evaluation:", refit["frozen_at_utc"])
print("Test evaluation completed:", test["test_evaluated_at_utc"])
print("Test matches:", test["match_counts"]["test"])
display(pd.DataFrame.from_dict(test["test_label_summary"]["status_counts"], orient="index", columns=["possessions"]))
source = test["sources"]["2026"]
display(pd.DataFrame([{
    "quarantined_quarters": len(source["quarantined_quarters"]),
    "quarantined_rows": source["quarantined_rows"],
    "verified_shot_markers": source["verified_retrospective_shot_markers"],
    "conflicting_shot_markers": source["conflicting_shot_markers"],
    "shot_conflict_possessions": source["shot_conflict_possessions"],
}]))

Frozen before test evaluation: 2026-10-11T13:09:59.484932+00:00
Test evaluation completed: 2026-10-11T13:10:05.743918+00:00
Test matches: 218


,possessions
eligible,51234
preprocessing_review,3629
no_pre_shot_action,29


,quarantined_quarters,quarantined_rows,verified_shot_markers,conflicting_shot_markers,shot_conflict_possessions
0,3,1616,1969,0,0


## 6. Held-out probability metrics

The comparison uses a constant probability learned from the combined
2023–2025 data as the reference. No test outcomes are used to fit that constant. These metrics
use possession weights. Lower BCE and Brier score are better; higher AP is
better. Test BCE excludes the L2 penalty.

The constant's test AP equals the weighted test prevalence because its scores
are tied, even though its probability was estimated from the development data.

In [6]:
metrics = pd.read_parquet(experiment / "test/metrics.parquet")
assert set(metrics.split) == {"test"}
display(metrics[["model", "examples", "possessions", "observed_shot_rate",
                 "log_loss", "brier_score", "average_precision"]].round(6))
print("Constant probability fitted on 2023–2025:", test["constant_probability"])

,model,examples,possessions,observed_shot_rate,log_loss,brier_score,average_precision
0,constant,290909,51234,0.216341,0.523087,0.169815,0.216341
1,logistic,290909,51234,0.216341,0.423148,0.134152,0.555929


Constant probability fitted on 2023–2025: 0.19969185162070446


## 7. Raw and weighted test confusion matrices

Rows show actual outcomes and columns show predictions, with a subsequent shot
as the positive class. Raw counts count action states. Weighted cells sum each
state's fraction of its possession weight and therefore need not be integers.
They are not counts of distinct possessions correctly classified.

False negatives and recall quantify missed shot-producing states; false positives
and specificity quantify errors on negative states. Accuracy alone obscures this trade-off. The threshold is fixed
at 0.5; it was not adjusted after viewing these test results.

In [7]:
confusion = pd.read_parquet(experiment / "test/confusion.parquet")
logistic = confusion.loc[confusion.model.eq("logistic")]
for row in logistic.itertuples(index=False):
    print(row.weighting, "| threshold:", row.threshold)
    matrix = pd.DataFrame([[row.tn, row.fp], [row.fn, row.tp]],
                          index=["Actual no shot", "Actual shot"],
                          columns=["Predict no shot", "Predict shot"])
    display(matrix.astype(int) if row.weighting == "unweighted" else matrix.round(2))
rates = ["accuracy", "precision", "recall", "specificity", "f1", "balanced_accuracy"]
print("Rates in percent:")
display((logistic.set_index("weighting")[rates].T * 100).round(2))
display(confusion.loc[confusion.weighting.eq("possession_weighted"), ["model", "threshold", *rates]].round(4))

unweighted | threshold: 0.5


,Predict no shot,Predict shot
Actual no shot,198638,10157
Actual shot,64642,17472


possession_weighted | threshold: 0.5


,Predict no shot,Predict shot
Actual no shot,38600.32,1549.68
Actual shot,7896.96,3187.04


Rates in percent:


weighting,unweighted,possession_weighted
accuracy,74.29,81.56
precision,63.24,67.28
recall,21.28,28.75
specificity,95.14,96.14
f1,31.84,40.29
balanced_accuracy,58.21,62.45


,model,threshold,accuracy,precision,recall,specificity,f1,balanced_accuracy
1,constant,0.5,0.7837,NaN,0.0000,1.0000,0.0000,0.5000
3,logistic,0.5,0.8156,0.6728,0.2875,0.9614,0.4029,0.6245


## 8. Calibration, coverage and limitations

Calibration bins compare average predicted probability with observed shot rate.
The bins are fixed and descriptive; no calibration model is fitted on test data.
The team coverage table records inclusion and exclusion counts for further
review without changing this evaluation.

The test result evaluates this selected model on this pinned 2026 sample. It
does not establish live-match reliability, causal action value or performance
on excluded episodes. Prefixes within a possession and match are correlated;
the report does not claim independent-observation confidence intervals.

In [8]:
calibration = pd.read_parquet(experiment / "test/calibration.parquet")
display(calibration.loc[calibration.model.eq("logistic"), [
    "bin_lower", "bin_upper", "examples", "weight_sum", "mean_prediction", "observed_shot_rate",
]].round(4))
print("Trial table:", "artifacts/logistic-tuned-2026/lambda_search.csv")
print("Frozen model:", "artifacts/logistic-tuned-2026/refit_model.pkl")
print("Final evaluation:", "artifacts/logistic-tuned-2026/test/")

,bin_lower,bin_upper,examples,weight_sum,mean_prediction,observed_shot_rate
1,0.0,0.1,105430,19872.8897,0.0484,0.0606
2,0.1,0.2,70917,11932.3102,0.1465,0.1661
3,0.2,0.3,40329,6550.4471,0.2465,0.2612
4,0.3,0.4,27121,4691.1814,0.3476,0.3325
5,0.4,0.5,19483,3450.4522,0.4477,0.4174
6,0.5,0.6,13034,2380.4738,0.5460,0.5444
7,0.6,0.7,8115,1391.9263,0.6455,0.7227
8,0.7,0.8,4860,815.4174,0.7430,0.9126
9,0.8,0.9,1532,143.6470,0.8281,0.9491
10,0.9,1.0,88,5.2548,0.9191,0.8975


Trial table: artifacts/logistic-tuned-2026/lambda_search.csv
Frozen model: artifacts/logistic-tuned-2026/refit_model.pkl
Final evaluation: artifacts/logistic-tuned-2026/test/
